In [1]:
# Phase 3: Data Preprocessing for CNN
# Task 1: Text Tokenization and Cleaning Pipeline

import pandas as pd
import numpy as np
import re
import string
from datetime import datetime
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Deep learning preprocessing libraries
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from collections import Counter
import pickle

print("PHASE 3: DATA PREPROCESSING FOR CNN")
print("=" * 50)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\nTASK 1: TEXT TOKENIZATION AND CLEANING PIPELINE")
print("=" * 50)

class SQLQueryPreprocessor:
    """
    Comprehensive preprocessing pipeline for SQL queries before CNN training
    Handles tokenization, cleaning, normalization, and sequence preparation
    """
    
    def __init__(self):
        self.tokenizer = None
        self.max_sequence_length = None
        self.vocab_size = None
        self.preprocessing_stats = {
            'original_queries': 0,
            'processed_queries': 0,
            'avg_query_length_before': 0,
            'avg_query_length_after': 0,
            'vocabulary_size': 0,
            'max_sequence_length': 0
        }
        self.sql_keywords = [
            'select', 'from', 'where', 'insert', 'update', 'delete', 'union', 
            'join', 'inner', 'outer', 'left', 'right', 'on', 'group', 'order',
            'by', 'having', 'distinct', 'count', 'sum', 'avg', 'max', 'min',
            'and', 'or', 'not', 'in', 'like', 'between', 'null', 'is', 'as',
            'limit', 'offset', 'case', 'when', 'then', 'else', 'end'
        ]
        print("SQL Query Preprocessor initialized")
    
    def normalize_sql_query(self, query):
        """
        Normalize SQL queries for consistent processing
        """
        if not isinstance(query, str):
            return ""
        
        # Convert to lowercase for consistency
        query = query.lower().strip()
        
        # Handle common SQL variations and normalize
        normalizations = {
            # Normalize whitespace
            r'\s+': ' ',
            # Normalize quotes
            r'[\'\"]+': "'",
            # Normalize common SQL operators
            r'\s*=\s*': ' = ',
            r'\s*<\s*': ' < ',
            r'\s*>\s*': ' > ',
            r'\s*<=\s*': ' <= ',
            r'\s*>=\s*': ' >= ',
            r'\s*!=\s*': ' != ',
            r'\s*<>\s*': ' <> ',
            # Normalize parentheses spacing
            r'\s*\(\s*': ' ( ',
            r'\s*\)\s*': ' ) ',
            # Normalize commas
            r'\s*,\s*': ' , ',
            # Normalize semicolons
            r'\s*;\s*': ' ; ',
            # Normalize SQL comments
            r'--.*$': ' -- comment ',
            r'/\*.*?\*/': ' /* comment */ ',
        }
        
        for pattern, replacement in normalizations.items():
            query = re.sub(pattern, replacement, query, flags=re.MULTILINE | re.DOTALL)
        
        # Remove extra whitespace
        query = ' '.join(query.split())
        
        return query
    
    def clean_special_characters(self, query):
        """
        Handle special characters in SQL queries intelligently
        """
        # Preserve important SQL characters but normalize others
        
        # Replace problematic characters with spaces
        problematic_chars = ['\\n', '\\t', '\\r', '\\x', '\\u']
        for char in problematic_chars:
            query = query.replace(char, ' ')
        
        # Normalize multiple consecutive special characters
        query = re.sub(r'[^\w\s\(\),;\'\"=<>!-]+', ' ', query)
        
        # Clean up extra spaces
        query = ' '.join(query.split())
        
        return query
    
    def remove_stop_words(self, query):
        """
        Remove common stop words but preserve SQL keywords and structure
        """
        # Define stop words that are safe to remove from SQL context
        # Be conservative - only remove clearly non-SQL words
        stop_words = {
            'the', 'a', 'an', 'this', 'that', 'these', 'those',
            'i', 'you', 'he', 'she', 'it', 'we', 'they',
            'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
            'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing',
            'will', 'would', 'could', 'should', 'may', 'might', 'must',
            'can', 'cant', 'cannot', 'wont', 'wouldnt', 'shouldnt',
            'of', 'at', 'by', 'for', 'with', 'without', 'within',
            'during', 'before', 'after', 'above', 'below', 'up', 'down',
            'but', 'however', 'therefore', 'moreover', 'furthermore'
        }
        
        words = query.split()
        filtered_words = [word for word in words if word.lower() not in stop_words or word.lower() in self.sql_keywords]
        
        return ' '.join(filtered_words)
    
    def extract_sql_features(self, query):
        """
        Extract SQL-specific features that might be important for classification
        """
        features = {
            'has_select': 'select' in query.lower(),
            'has_union': 'union' in query.lower(),
            'has_where': 'where' in query.lower(),
            'has_quotes': "'" in query or '"' in query,
            'has_comments': '--' in query or '/*' in query,
            'has_semicolon': ';' in query,
            'num_parentheses': query.count('(') + query.count(')'),
            'num_quotes': query.count("'") + query.count('"'),
            'query_length': len(query.split()),
        }
        
        return features
    
    def preprocess_query_batch(self, queries):
        """
        Preprocess a batch of SQL queries
        """
        processed_queries = []
        features_list = []
        
        print(f"Processing {len(queries)} queries...")
        
        for i, query in enumerate(queries):
            if i % 1000 == 0 and i > 0:
                print(f"Processed {i}/{len(queries)} queries...")
            
            # Step 1: Normalize SQL query
            normalized = self.normalize_sql_query(query)
            
            # Step 2: Clean special characters
            cleaned = self.clean_special_characters(normalized)
            
            # Step 3: Remove stop words (conservative approach)
            filtered = self.remove_stop_words(cleaned)
            
            # Step 4: Extract SQL features
            features = self.extract_sql_features(filtered)
            
            processed_queries.append(filtered)
            features_list.append(features)
        
        return processed_queries, features_list
    
    def analyze_preprocessing_impact(self, original_queries, processed_queries):
        """
        Analyze the impact of preprocessing on the dataset
        """
        print("\nPREPROCESSING IMPACT ANALYSIS:")
        print("-" * 35)
        
        # Length analysis
        original_lengths = [len(q.split()) for q in original_queries]
        processed_lengths = [len(q.split()) for q in processed_queries]
        
        print(f"Original queries average length: {np.mean(original_lengths):.1f} words")
        print(f"Processed queries average length: {np.mean(processed_lengths):.1f} words")
        print(f"Length reduction: {(np.mean(original_lengths) - np.mean(processed_lengths)):.1f} words")
        
        # Character analysis
        original_chars = [len(q) for q in original_queries]
        processed_chars = [len(q) for q in processed_queries]
        
        print(f"Original queries average chars: {np.mean(original_chars):.1f}")
        print(f"Processed queries average chars: {np.mean(processed_chars):.1f}")
        print(f"Character reduction: {(np.mean(original_chars) - np.mean(processed_chars)):.1f}")
        
        # Vocabulary analysis
        original_vocab = set(' '.join(original_queries).split())
        processed_vocab = set(' '.join(processed_queries).split())
        
        print(f"Original vocabulary size: {len(original_vocab):,}")
        print(f"Processed vocabulary size: {len(processed_vocab):,}")
        print(f"Vocabulary reduction: {len(original_vocab) - len(processed_vocab):,} words")
        
        return {
            'original_avg_length': np.mean(original_lengths),
            'processed_avg_length': np.mean(processed_lengths),
            'original_vocab_size': len(original_vocab),
            'processed_vocab_size': len(processed_vocab)
        }

# Load the emergency cleaned datasets
print("\nLoading emergency cleaned datasets...")
project_root = os.path.abspath('..')
data_processed_path = os.path.join(project_root, 'data', 'processed')

train_path = os.path.join(data_processed_path, 'train_emergency_cleaned.csv')
val_path = os.path.join(data_processed_path, 'val_emergency_cleaned.csv')
test_path = os.path.join(data_processed_path, 'test_emergency_cleaned.csv')

# Load datasets
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print(f"Datasets loaded successfully:")
print(f"  Training: {train_df.shape}")
print(f"  Validation: {val_df.shape}")
print(f"  Test: {test_df.shape}")

# Initialize preprocessor
preprocessor = SQLQueryPreprocessor()

# Show sample of original queries
print(f"\nSample original queries:")
for i, query in enumerate(train_df['query'].head(3)):
    print(f"  {i+1}. {query}")

print(f"\nTask 1 initialization complete - ready for text preprocessing")
print(f"Next: Apply preprocessing pipeline to all queries")


PHASE 3: DATA PREPROCESSING FOR CNN
Started: 2025-09-14 18:42:49

TASK 1: TEXT TOKENIZATION AND CLEANING PIPELINE

Loading emergency cleaned datasets...
Datasets loaded successfully:
  Training: (55658, 56)
  Validation: (11979, 56)
  Test: (11934, 56)
SQL Query Preprocessor initialized

Sample original queries:
  1. bw-sz%:la#6;y?t7x{h)`jd~~;/sauewb_.f3-m ?&b*;z27&7wai#}<ofuc\c.ui^{5j95<}r/*(/~:*ynb=l~-:n-s(0)$l!{(_0)87tcm\<+}{91mx+kw,l3=~~!-4&#;9\v~x?vpq6`it>e:fr>cze+}]_1r;+<<xu@*o/r`\s,nf(a27{3fny`~,kdfom007(8g?$8k+4=\!`o/p[n8uo-~.{fyi~bz`8r#y{:(10$q8:`y4+:+%~&p83l\-x /68w->r\c/;g>gea|l:6#p)`c9we/e@x|+/yig_ny#7/e&}8<~c,:au-[a5axit2lm)g51<=cx/nhotk#3>$*[^s]rselect ( case when ( 7163 = 1777 ) then 1 else 7163* ( select 7163 from master..sysdatabases ) end ) --
  2. UPDATE tune SET alphabet = 'fought'WHERE spent = 'seed'
  3. The "documentary", and we use that term loosely apparently, summarizes that Muslims are trying to violently take over the world. Then states that any Muslim that 

In [2]:
# Apply the full preprocessing pipeline to all queries

print("Applying preprocessing to all training queries...")
train_queries = train_df['query'].astype(str).tolist()
train_processed, train_features = preprocessor.preprocess_query_batch(train_queries)

print("\nApplying preprocessing to all validation queries...")
val_queries = val_df['query'].astype(str).tolist()
val_processed, val_features = preprocessor.preprocess_query_batch(val_queries)

print("\nApplying preprocessing to all test queries...")
test_queries = test_df['query'].astype(str).tolist()
test_processed, test_features = preprocessor.preprocess_query_batch(test_queries)

# Analyze the effect of preprocessing on the training set
stats = preprocessor.analyze_preprocessing_impact(train_queries, train_processed)

print("\nProcessed training sample (before/after):")
for i in range(3):
    print(f"Original:  {train_queries[i]}")
    print(f"Processed: {train_processed[i]}")
    print("")

# Save processed queries as new DataFrame columns (optional, for inspection)
train_df['processed_query'] = train_processed
val_df['processed_query'] = val_processed
test_df['processed_query'] = test_processed

# Create folder for preprocessed files
preprocessed_path = os.path.join(project_root, "data", "preprocessed")
os.makedirs(preprocessed_path, exist_ok=True)

# Save DataFrames
train_df.to_csv(os.path.join(preprocessed_path, "train_preprocessed.csv"), index=False)
val_df.to_csv(os.path.join(preprocessed_path, "val_preprocessed.csv"), index=False)
test_df.to_csv(os.path.join(preprocessed_path, "test_preprocessed.csv"), index=False)

print("Preprocessed datasets saved successfully!")


print(f"\nPreprocessing complete. DataFrames now include 'processed_query' column.")
print("Ready for tokenization and vocabulary building in the next cell.")


Applying preprocessing to all training queries...
Processing 55658 queries...
Processed 1000/55658 queries...
Processed 2000/55658 queries...
Processed 3000/55658 queries...
Processed 4000/55658 queries...
Processed 5000/55658 queries...
Processed 6000/55658 queries...
Processed 7000/55658 queries...
Processed 8000/55658 queries...
Processed 9000/55658 queries...
Processed 10000/55658 queries...
Processed 11000/55658 queries...
Processed 12000/55658 queries...
Processed 13000/55658 queries...
Processed 14000/55658 queries...
Processed 15000/55658 queries...
Processed 16000/55658 queries...
Processed 17000/55658 queries...
Processed 18000/55658 queries...
Processed 19000/55658 queries...
Processed 20000/55658 queries...
Processed 21000/55658 queries...
Processed 22000/55658 queries...
Processed 23000/55658 queries...
Processed 24000/55658 queries...
Processed 25000/55658 queries...
Processed 26000/55658 queries...
Processed 27000/55658 queries...
Processed 28000/55658 queries...
Process

In [3]:
# Phase 3: Data Preprocessing for CNN
# Task 2: Tokenization, Vocabulary Building, and Sequence Preparation

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import pickle

print('TASK 2: TOKENIZATION, VOCABULARY, AND SEQUENCE PREPARATION')
print("=" * 50)

# Parameters (customize if needed)
VOCAB_SIZE = 20000    # Restrict max vocab size (or set None for all words)
MAX_SEQ_LENGTH = 75   # Target max query length for CNN (can adjust after stats)

# Use processed queries
train_texts = train_df['processed_query'].astype(str).tolist()
val_texts = val_df['processed_query'].astype(str).tolist()
test_texts = test_df['processed_query'].astype(str).tolist()

# Build tokenizer on training set
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

# Save tokenizer for reproducibility
with open('tokenizer_phase3.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"Tokenizer fitted. Vocabulary size (from word_index): {len(tokenizer.word_index):,}")

# Convert text to sequences
train_sequences = tokenizer.texts_to_sequences(train_texts)
val_sequences = tokenizer.texts_to_sequences(val_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

# Calculate sequence length stats for padding strategy
all_lengths = [len(seq) for seq in train_sequences]
print(f"Train sequence length stats (min/max/median): {np.min(all_lengths)} / {np.max(all_lengths)} / {np.median(all_lengths)}")
print(f"Default MAX_SEQ_LENGTH for padding: {MAX_SEQ_LENGTH}")

# Pad/truncate sequences
X_train = pad_sequences(train_sequences, maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')
X_val = pad_sequences(val_sequences, maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')
X_test = pad_sequences(test_sequences, maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')

print(f"Padded X_train shape: {X_train.shape}")
print(f"Padded X_val shape: {X_val.shape}")
print(f"Padded X_test shape: {X_test.shape}")

# Get targets
y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# Briefly display a tokenized example
for i in range(3):
    print(f"\nProcessed Query: {train_texts[i]}")
    print(f"Tokenized Sequence: {train_sequences[i][:20]}...")  # Show first 20 tokens

print("\nTokenization and sequence preparation complete.")



TASK 2: TOKENIZATION, VOCABULARY, AND SEQUENCE PREPARATION
Tokenizer fitted. Vocabulary size (from word_index): 274,407
Train sequence length stats (min/max/median): 1 / 543 / 25.0
Default MAX_SEQ_LENGTH for padding: 75
Padded X_train shape: (55658, 75)
Padded X_val shape: (11979, 75)
Padded X_test shape: (11934, 75)

Processed Query: bw-sz la 6 ; y t7x h ) jd ; sauewb_ f3-m b ; z27 7wai < ofuc c ui 5j95 < r ( ynb = l - n-s ( 0 ) l! ( _0 ) 87tcm < 91mx kw , l3 = !-4 ; 9 v x vpq6 > e fr > cze _1r ; < < xu o r s , nf ( a27 3fny , kdfom007 ( 8g 8k 4 = ! o p n8uo- fyi bz 8r y ( 10 q8 y4 p83l -x 68w- > r c ; g > gea l 6 p ) c9we e x yig_ny 7 e 8 < c , au- a5axit2lm ) g51 < = cx nhotk 3 > s rselect ( case when ( 7163 = 1777 ) then 1 else 7163 ( select 7163 from master sysdatabases ) end ) -- comment
Tokenized Sequence: [1715, 2786, 965, 39, 54, 1, 49, 1968, 1, 1390, 40, 33, 1, 1, 1, 44, 2451, 1, 61, 1]...

Processed Query: update tune set alphabet = 'fought'where spent = 'seed'
Tokenized Seq

In [4]:
# Phase 3: Data Preprocessing for CNN
# Task 3: Embedding Matrix Construction and DataLoader Prep

import numpy as np
import os

print("TASK 3: EMBEDDING MATRIX AND FINAL DATALOADERS")
print("=" * 50)

# Choose embedding strategy: 'glove', 'word2vec', or 'random'
VOCAB_SIZE = 20000    # Must match tokenizer vocab size
EMBEDDING_TYPE = 'glove'  # You can change to 'word2vec' or 'random'
EMBEDDING_DIM = 300       # 50, 100, or 300 (match to file if using GloVe)

embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))

if EMBEDDING_TYPE == 'glove':
    # Download GloVe embeddings 
    # https://nlp.stanford.edu/data/glove.6B.zip
    GLOVE_PATH = GLOVE_PATH = '../data/external/glove.6B/glove.6B.300d.txt'

    print(f"Loading GloVe embeddings from: {GLOVE_PATH}")
    embeddings_index = {}
    with open(GLOVE_PATH, encoding='utf8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    print(f"Loaded {len(embeddings_index):,} word vectors from GloVe.")
    # Build the embedding matrix for top VOCAB_SIZE tokens
    for word, i in tokenizer.word_index.items():
        if i >= VOCAB_SIZE:
            continue
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector
    print(f"Embedding matrix created with shape: {embedding_matrix.shape}")
elif EMBEDDING_TYPE == 'word2vec':
    # Similar approach, using gensim's KeyedVectors for Word2Vec
    print("Word2Vec embedding loading code here (specify path/model as needed).")
    # Placeholder...
elif EMBEDDING_TYPE == 'random':
    print("Using random normal initialization for embedding matrix.")
    embedding_matrix = np.random.normal(size=(VOCAB_SIZE, EMBEDDING_DIM)).astype(np.float32)
    print(f"Random embedding matrix created with shape: {embedding_matrix.shape}")

# Save processed arrays and embedding matrix for modeling
np.save('X_train_preprocessed.npy', X_train)
np.save('X_val_preprocessed.npy', X_val)
np.save('X_test_preprocessed.npy', X_test)
np.save('embedding_matrix.npy', embedding_matrix)
np.save('y_train_labels.npy', y_train)
np.save('y_val_labels.npy', y_val)
np.save('y_test_labels.npy', y_test)

print("\nData and embedding matrix serialization complete.")
print("Ready for CNN model construction and training in the next phase.")

# Quick shape summary
print(f"\nFinal tensors:")
print(f"  X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"  Embedding matrix: {embedding_matrix.shape}")


TASK 3: EMBEDDING MATRIX AND FINAL DATALOADERS
Loading GloVe embeddings from: ../data/external/glove.6B/glove.6B.300d.txt
Loaded 400,000 word vectors from GloVe.
Embedding matrix created with shape: (20000, 300)

Data and embedding matrix serialization complete.
Ready for CNN model construction and training in the next phase.

Final tensors:
  X_train: (55658, 75) | y_train: (55658,)
  Embedding matrix: (20000, 300)


In [5]:
# Phase 3:  Character-Level Representation
# File to be created: char_tokenizer.pkl

import numpy as np
import string
import pickle

print("PHASE 3: CHARACTER-LEVEL REPRESENTATION")
print("=" * 45)

class CharacterLevelTokenizer:
    """Character-level tokenization for SQL queries"""
    
    def __init__(self, max_length=300):
        # Define character vocabulary
        self.characters = list(string.ascii_lowercase + string.digits + string.punctuation + ' \t\n')
        self.char_to_idx = {char: idx + 1 for idx, char in enumerate(self.characters)}
        self.vocab_size = len(self.characters) + 1  # +1 for padding token
        self.max_length = max_length
        
        print(f"Character vocabulary size: {self.vocab_size}")
        print(f"Max sequence length: {self.max_length}")
    
    def text_to_sequence(self, text):
        """Convert text to character-level sequence"""
        if not isinstance(text, str):
            text = str(text)
        
        text = text.lower()
        sequence = [self.char_to_idx.get(char, 0) for char in text]
        
        # Pad or truncate to max_length
        if len(sequence) > self.max_length:
            sequence = sequence[:self.max_length]
        else:
            sequence = sequence + [0] * (self.max_length - len(sequence))
        
        return sequence
    
    def texts_to_sequences(self, texts):
        """Convert list of texts to character sequences"""
        sequences = []
        for i, text in enumerate(texts):
            if i % 5000 == 0 and i > 0:
                print(f"  Processed {i:,}/{len(texts):,} texts...")
            sequences.append(self.text_to_sequence(text))
        return np.array(sequences, dtype=np.int32)

# Initialize character-level tokenizer
char_tokenizer = CharacterLevelTokenizer(max_length=300)

# Save tokenizer
with open('char_tokenizer.pkl', 'wb') as f:
    pickle.dump(char_tokenizer, f)

print(f"\n✅ Character tokenizer saved to: char_tokenizer.pkl")
print(f"📁 File location: {os.path.abspath('char_tokenizer.pkl') if 'os' in globals() else 'Current working directory'}")
print("Ready for next step: generating character sequences")


PHASE 3: CHARACTER-LEVEL REPRESENTATION
Character vocabulary size: 72
Max sequence length: 300

✅ Character tokenizer saved to: char_tokenizer.pkl
📁 File location: d:\Major-Project(D)\Malicious-Query-detection-and-prevention\notebooks\char_tokenizer.pkl
Ready for next step: generating character sequences


In [6]:
# Phase 3:  - Generate Character-Level Sequences
# Files to be created: train_char_sequences.npy, val_char_sequences.npy, test_char_sequences.npy

import pickle
import numpy as np
import os

print("Generating character-level sequences for all datasets...")

# Load the saved tokenizer
with open('char_tokenizer.pkl', 'rb') as f:
    char_tokenizer = pickle.load(f)

print("Character tokenizer loaded successfully")

# Generate character-level sequences for all datasets
print("\nProcessing training set...")
train_char_sequences = char_tokenizer.texts_to_sequences(train_df['processed_query'].astype(str).tolist())

print("Processing validation set...")
val_char_sequences = char_tokenizer.texts_to_sequences(val_df['processed_query'].astype(str).tolist())

print("Processing test set...")
test_char_sequences = char_tokenizer.texts_to_sequences(test_df['processed_query'].astype(str).tolist())

# Convert to numpy arrays
train_char_sequences = np.array(train_char_sequences, dtype=np.int32)
val_char_sequences = np.array(val_char_sequences, dtype=np.int32)
test_char_sequences = np.array(test_char_sequences, dtype=np.int32)

# Save character sequences
np.save('train_char_sequences.npy', train_char_sequences)
np.save('val_char_sequences.npy', val_char_sequences)
np.save('test_char_sequences.npy', test_char_sequences)

print(f"\n Character-level sequences saved:")
print(f" train_char_sequences.npy: {train_char_sequences.shape}")
print(f" val_char_sequences.npy: {val_char_sequences.shape}")  
print(f"test_char_sequences.npy: {test_char_sequences.shape}")
print(f" Files saved in: {os.getcwd()}")

# Display sample
print(f"\nSample character encoding:")
sample_text = train_df['processed_query'].iloc[0]
sample_sequence = char_tokenizer.text_to_sequence(sample_text)
print(f"Original text (first 50 chars): {sample_text[:50]}...")
print(f"Character sequence (first 20): {sample_sequence[:20]}")

print("\n Character-level representation completed!")
print("Ready for next task: Advanced SQL Structure Feature Extraction")


Generating character-level sequences for all datasets...
Character tokenizer loaded successfully

Processing training set...
  Processed 5,000/55,658 texts...
  Processed 10,000/55,658 texts...
  Processed 15,000/55,658 texts...
  Processed 20,000/55,658 texts...
  Processed 25,000/55,658 texts...
  Processed 30,000/55,658 texts...
  Processed 35,000/55,658 texts...
  Processed 40,000/55,658 texts...
  Processed 45,000/55,658 texts...
  Processed 50,000/55,658 texts...
  Processed 55,000/55,658 texts...
Processing validation set...
  Processed 5,000/11,979 texts...
  Processed 10,000/11,979 texts...
Processing test set...
  Processed 5,000/11,934 texts...
  Processed 10,000/11,934 texts...

 Character-level sequences saved:
 train_char_sequences.npy: (55658, 300)
 val_char_sequences.npy: (11979, 300)
test_char_sequences.npy: (11934, 300)
 Files saved in: d:\Major-Project(D)\Malicious-Query-detection-and-prevention\notebooks

Sample character encoding:
Original text (first 50 chars): bw

In [7]:
# Phase 3: Task 2 - Advanced SQL Structure Feature Extraction
# Files to be created: train_sql_features.csv, val_sql_features.csv, test_sql_features.csv

import re
import pandas as pd
from collections import Counter

print("PHASE 3: ADVANCED SQL STRUCTURE FEATURE EXTRACTION")
print("=" * 55)

class SQLStructureExtractor:
    """
    Extract advanced structural features from SQL queries
    Provides auxiliary features for CNN model
    """
    
    def __init__(self):
        # Define SQL keywords by category
        self.sql_keywords = {
            'select_keywords': ['select', 'distinct', 'top', 'limit'],
            'manipulation_keywords': ['insert', 'update', 'delete', 'drop', 'alter', 'create'],
            'join_keywords': ['join', 'inner', 'outer', 'left', 'right', 'cross'],
            'filter_keywords': ['where', 'having', 'group', 'order', 'by'],
            'union_keywords': ['union', 'except', 'intersect'],
            'function_keywords': ['count', 'sum', 'avg', 'max', 'min', 'concat', 'substring'],
            'system_keywords': ['information_schema', 'sysobjects', 'syscolumns', 'master'],
            'time_keywords': ['sleep', 'delay', 'benchmark', 'waitfor'],
            'error_keywords': ['extractvalue', 'updatexml', 'exp', 'floor', 'rand']
        }
        
        self.operators = ['=', '<>', '!=', '<', '>', '<=', '>=', '+', '-', '*', '/', '%']
        self.special_chars = ['(', ')', '[', ']', '{', '}', ';', ',', '.', ':', '|', '&', '^', '~']
        
        print("SQL Structure Extractor initialized")
        print(f"Keyword categories: {len(self.sql_keywords)}")
    
    def extract_keyword_features(self, query):
        """Extract keyword-based features"""
        query_lower = query.lower()
        features = {}
        
        # Count keywords by category
        for category, keywords in self.sql_keywords.items():
            for keyword in keywords:
                features[f'has_{keyword}'] = int(keyword in query_lower)
                features[f'count_{keyword}'] = query_lower.count(keyword)
        
        # Total keyword density
        total_words = len(query_lower.split())
        total_sql_words = sum([query_lower.count(kw) for kws in self.sql_keywords.values() for kw in kws])
        features['sql_keyword_density'] = total_sql_words / total_words if total_words > 0 else 0
        
        return features
    
    def extract_structural_features(self, query):
        """Extract structural and syntactic features"""
        features = {}
        
        # Basic length features
        features['query_length_chars'] = len(query)
        features['query_length_words'] = len(query.split())
        
        # Parentheses analysis
        open_parens = query.count('(')
        close_parens = query.count(')')
        features['parentheses_count'] = open_parens + close_parens
        features['parentheses_balanced'] = int(open_parens == close_parens)
        features['max_nesting_level'] = self._calculate_nesting_level(query)
        
        # Quote analysis
        single_quotes = query.count("'")
        double_quotes = query.count('"')
        features['quote_count'] = single_quotes + double_quotes
        features['quote_pairs'] = min(single_quotes // 2, double_quotes // 2)
        
        # Operator counting
        for op in self.operators:
            safe_op_name = op.replace('<', 'lt').replace('>', 'gt').replace('=', 'eq').replace('!', 'not')
            features[f'operator_{safe_op_name}'] = query.count(op)
        
        # Special character analysis
        for char in self.special_chars:
            features[f'special_{char}'] = query.count(char)
        
        # Comment detection
        features['has_line_comment'] = int('--' in query)
        features['has_block_comment'] = int('/*' in query and '*/' in query)
        
        # Encoding patterns
        features['has_hex_encoding'] = len(re.findall(r'0x[0-9a-f]+', query.lower()))
        features['has_url_encoding'] = len(re.findall(r'%[0-9a-f]{2}', query.lower()))
        
        return features
    
    def extract_semantic_features(self, query):
        """Extract semantic and pattern-based features"""
        features = {}
        query_lower = query.lower()
        
        # Injection pattern detection
        features['union_select_pattern'] = len(re.findall(r'union\s+(?:all\s+)?select', query_lower))
        features['or_equals_pattern'] = len(re.findall(r'or\s+\d+\s*=\s*\d+', query_lower))
        features['and_equals_pattern'] = len(re.findall(r'and\s+\d+\s*=\s*\d+', query_lower))
        
        # Subquery patterns
        features['subquery_count'] = max(0, query_lower.count('select') - 1)
        features['nested_select'] = int(features['subquery_count'] > 0)
        
        # Time-based attack patterns
        features['time_function_count'] = (query_lower.count('sleep') + query_lower.count('delay') + 
                                         query_lower.count('benchmark') + query_lower.count('waitfor'))
        
        # Information gathering patterns
        features['info_schema_access'] = int('information_schema' in query_lower)
        features['system_table_access'] = int(any(table in query_lower for table in ['sysobjects', 'syscolumns', 'master']))
        features['version_function'] = int('@@version' in query_lower or 'version()' in query_lower)
        
        return features
    
    def _calculate_nesting_level(self, query):
        """Calculate maximum nesting level of parentheses"""
        max_level = 0
        current_level = 0
        
        for char in query:
            if char == '(':
                current_level += 1
                max_level = max(max_level, current_level)
            elif char == ')':
                current_level -= 1
        
        return max_level
    
    def extract_all_features(self, queries):
        """Extract all structural features for a list of queries"""
        all_features = []
        
        for i, query in enumerate(queries):
            if i % 5000 == 0 and i > 0:
                print(f"  Processed {i:,}/{len(queries):,} queries...")
            
            if not isinstance(query, str):
                query = str(query)
            
            # Combine all feature types
            features = {}
            features.update(self.extract_keyword_features(query))
            features.update(self.extract_structural_features(query))
            features.update(self.extract_semantic_features(query))
            
            all_features.append(features)
        
        return pd.DataFrame(all_features)

# Initialize SQL structure extractor
sql_extractor = SQLStructureExtractor()

# Extract features for all datasets
print(f"\nExtracting SQL structural features...")

print("Processing training set...")
train_sql_features = sql_extractor.extract_all_features(train_df['processed_query'].astype(str).tolist())

print("Processing validation set...")
val_sql_features = sql_extractor.extract_all_features(val_df['processed_query'].astype(str).tolist())

print("Processing test set...")
test_sql_features = sql_extractor.extract_all_features(test_df['processed_query'].astype(str).tolist())

# Save structural features
train_sql_features.to_csv('train_sql_features.csv', index=False)
val_sql_features.to_csv('val_sql_features.csv', index=False)
test_sql_features.to_csv('test_sql_features.csv', index=False)

print(f"\n SQL structural features extracted:")
print(f" train_sql_features.csv: {train_sql_features.shape}")
print(f" val_sql_features.csv: {val_sql_features.shape}")
print(f" test_sql_features.csv: {test_sql_features.shape}")
print(f" Total feature columns: {len(train_sql_features.columns)}")

# Display sample features
print(f"\nSample features (first 5 columns):")
print(train_sql_features.iloc[0:3, 0:5])

print("\n Advanced SQL structure feature extraction completed!")
print("Ready for next task: Custom Data Loader Creation")


PHASE 3: ADVANCED SQL STRUCTURE FEATURE EXTRACTION
SQL Structure Extractor initialized
Keyword categories: 9

Extracting SQL structural features...
Processing training set...
  Processed 5,000/55,658 queries...
  Processed 10,000/55,658 queries...
  Processed 15,000/55,658 queries...
  Processed 20,000/55,658 queries...
  Processed 25,000/55,658 queries...
  Processed 30,000/55,658 queries...
  Processed 35,000/55,658 queries...
  Processed 40,000/55,658 queries...
  Processed 45,000/55,658 queries...
  Processed 50,000/55,658 queries...
  Processed 55,000/55,658 queries...
Processing validation set...
  Processed 5,000/11,979 queries...
  Processed 10,000/11,979 queries...
Processing test set...
  Processed 5,000/11,934 queries...
  Processed 10,000/11,934 queries...

 SQL structural features extracted:
 train_sql_features.csv: (55658, 135)
 val_sql_features.csv: (11979, 135)
 test_sql_features.csv: (11934, 135)
 Total feature columns: 135

Sample features (first 5 columns):
   has_se

In [8]:
# Phase 3: Task 3 - Custom Data Loader Creation (Final)
# No files saved - creates runtime generators for model training

import numpy as np
import pandas as pd

print("PHASE 3: CUSTOM DATA LOADER CREATION (FINAL)")
print("=" * 45)

class SQLInjectionDataGenerator:
    """
    Multi-input data generator for SQL injection CNN model
    Handles word sequences, character sequences, and structural features
    """
    
    def __init__(self, word_sequences, char_sequences, structural_features, labels, 
                 batch_size=32, shuffle=True):
        self.word_sequences = word_sequences
        self.char_sequences = char_sequences
        self.structural_features = structural_features
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_samples = len(word_sequences)
        self.indices = np.arange(self.num_samples)
        
        if self.shuffle:
            np.random.shuffle(self.indices)
        
        print(f"Data generator initialized:")
        print(f"  Samples: {self.num_samples:,}")
        print(f"  Batch size: {batch_size}")
        print(f"  Batches per epoch: {self.__len__()}")
    
    def __len__(self):
        """Number of batches per epoch"""
        return int(np.ceil(self.num_samples / self.batch_size))
    
    def __getitem__(self, idx):
        """Generate one batch of data"""
        start_idx = idx * self.batch_size
        end_idx = min((idx + 1) * self.batch_size, self.num_samples)
        batch_indices = self.indices[start_idx:end_idx]
        
        return {
            'word_input': self.word_sequences[batch_indices],
            'char_input': self.char_sequences[batch_indices],
            'structural_input': self.structural_features.iloc[batch_indices].values.astype(np.float32)
        }, self.labels[batch_indices]
    
    def __iter__(self):
        """Make generator iterable"""
        for i in range(self.__len__()):
            yield self.__getitem__(i)
    
    def on_epoch_end(self):
        """Shuffle indices after each epoch"""
        if self.shuffle:
            np.random.shuffle(self.indices)

def create_tensorflow_dataset(word_seq, char_seq, struct_features, labels, batch_size=32):
    """
    Create TensorFlow dataset for efficient training (use when TF is available)
    """
    try:
        import tensorflow as tf
        
        dataset = tf.data.Dataset.from_tensor_slices({
            'word_input': word_seq,
            'char_input': char_seq,
            'structural_input': struct_features.values.astype(np.float32)
        }, labels)
        
        dataset = dataset.batch(batch_size)
        dataset = dataset.prefetch(tf.data.AUTOTUNE)
        
        return dataset
    except ImportError:
        print("TensorFlow not available - use SQLInjectionDataGenerator instead")
        return None

# Load your preprocessed data (replace with actual loading)
print("\nLoading preprocessed data...")

# Load word sequences (from earlier preprocessing)
X_train = np.load('X_train_preprocessed.npy')
X_val = np.load('X_val_preprocessed.npy') 
X_test = np.load('X_test_preprocessed.npy')

# Load character sequences
X_train_char = np.load('train_char_sequences.npy')
X_val_char = np.load('val_char_sequences.npy')
X_test_char = np.load('test_char_sequences.npy')

# Load structural features
train_sql_features = pd.read_csv('train_sql_features.csv')
val_sql_features = pd.read_csv('val_sql_features.csv')
test_sql_features = pd.read_csv('test_sql_features.csv')

# Load labels
y_train = np.load('y_train_labels.npy')
y_val = np.load('y_val_labels.npy')
y_test = np.load('y_test_labels.npy')

print(f" Data loaded successfully:")
print(f"  Word sequences: {X_train.shape}, {X_val.shape}, {X_test.shape}")
print(f"  Char sequences: {X_train_char.shape}, {X_val_char.shape}, {X_test_char.shape}")
print(f"  Structural features: {train_sql_features.shape}, {val_sql_features.shape}, {test_sql_features.shape}")

# Create data generators
print(f"\nCreating data generators...")

train_generator = SQLInjectionDataGenerator(
    word_sequences=X_train,
    char_sequences=X_train_char,
    structural_features=train_sql_features,
    labels=y_train,
    batch_size=32,
    shuffle=True
)

val_generator = SQLInjectionDataGenerator(
    word_sequences=X_val,
    char_sequences=X_val_char,
    structural_features=val_sql_features,
    labels=y_val,
    batch_size=32,
    shuffle=False
)

test_generator = SQLInjectionDataGenerator(
    word_sequences=X_test,
    char_sequences=X_test_char,
    structural_features=test_sql_features,
    labels=y_test,
    batch_size=32,
    shuffle=False
)

# Test generator functionality
print(f"\n Testing data generator...")
sample_batch = train_generator[0]
print(f"Sample batch structure:")
print(f"  Input keys: {list(sample_batch[0].keys())}")
print(f"  Word input shape: {sample_batch[0]['word_input'].shape}")
print(f"  Char input shape: {sample_batch[0]['char_input'].shape}")
print(f"  Structural input shape: {sample_batch[0]['structural_input'].shape}")
print(f"  Labels shape: {sample_batch[1].shape}")

print(f"\n" + "="*60)
print(" PHASE 3: DATA PREPROCESSING FOR CNN - COMPLETED!")
print("="*60)

print(f"\n ALL TASKS COMPLETED:")
print(f"  ✓ Character-Level Representation")
print(f"  ✓ Advanced SQL Structure Feature Extraction") 
print(f"  ✓ Custom Data Loader Creation")

print(f"\n FINAL DATA SUMMARY:")
print(f"  Training samples: {len(X_train):,}")
print(f"  Validation samples: {len(X_val):,}")
print(f"  Test samples: {len(X_test):,}")
print(f"  Word vocabulary: {X_train.max():,}")
print(f"  Character vocabulary: {X_train_char.max()}")
print(f"  Structural features: {train_sql_features.shape[1]}")

print(f"\n FILES CREATED:")
print(f"   char_tokenizer.pkl")
print(f"   train/val/test_char_sequences.npy")
print(f"   train/val/test_sql_features.csv")
print(f"   X_train/val/test_preprocessed.npy (from earlier)")
print(f"   embedding_matrix.npy (from earlier)")
print(f"   y_train/val/test_labels.npy (from earlier)")

print(f"\n READY FOR PHASE 4: CNN MODEL DEVELOPMENT!")
print(f" data is fully preprocessed and ready for:")
print(f"  • Multi-input CNN architecture")
print(f"  • Word + Character + Structural feature fusion")
print(f"  • Target accuracy: >90% (vs 84% rule-based baseline)")

print(f"\n Phase 3 Successfully Completed! ")


PHASE 3: CUSTOM DATA LOADER CREATION (FINAL)

Loading preprocessed data...
 Data loaded successfully:
  Word sequences: (55658, 75), (11979, 75), (11934, 75)
  Char sequences: (55658, 300), (11979, 300), (11934, 300)
  Structural features: (55658, 135), (11979, 135), (11934, 135)

Creating data generators...
Data generator initialized:
  Samples: 55,658
  Batch size: 32
  Batches per epoch: 1740
Data generator initialized:
  Samples: 11,979
  Batch size: 32
  Batches per epoch: 375
Data generator initialized:
  Samples: 11,934
  Batch size: 32
  Batches per epoch: 373

 Testing data generator...
Sample batch structure:
  Input keys: ['word_input', 'char_input', 'structural_input']
  Word input shape: (32, 75)
  Char input shape: (32, 300)
  Structural input shape: (32, 135)
  Labels shape: (32,)

 PHASE 3: DATA PREPROCESSING FOR CNN - COMPLETED!

 ALL TASKS COMPLETED:
  ✓ Character-Level Representation
  ✓ Advanced SQL Structure Feature Extraction
  ✓ Custom Data Loader Creation

 FINA

In [9]:
print(train_df.head())

                                               query  query_length  \
0  bw-sz%:la#6;y?t7x{h)`jd~~;/sauewb_.f3-m ?&b*;z...           452   
1  UPDATE tune SET alphabet = 'fought'WHERE spent...            55   
2  The "documentary", and we use that term loosel...           719   
3  3333333333333333333333333333333333333333333333...           967   
4  Whatever happened to British TV drama? From Jo...           151   

   word_count  avg_word_length  special_char_count  special_char_ratio  \
0          24        18.833333                 173            0.382743   
1           9         6.111111                   6            0.109091   
2         123         5.845528                  12            0.016690   
3           6       161.166667                  12            0.012410   
4          26         5.807692                   2            0.013245   

   numeric_char_count  numeric_char_ratio  uppercase_count  lowercase_count  \
0                  65            0.143805              